# 🎬 ProXMax — স্বয়ংক্রিয় English → Bangla ভিডিও ডাবিং

**Google Drive**-এর `/content/drive/MyDrive/Video/input` ফোল্ডারের ইংরেজি ভিডিও বাংলায় ডাব করে `/content/drive/MyDrive/Video/output`-এ সেভ করে।

| ধাপ | প্রযুক্তি ও সীমা |
|---|---|
| ১. ইংরেজি STT | `faster-whisper`: `distil-large-v3` ডিফল্ট (শুধু ইংরেজি), `large-v3` বিকল্প; hotwords ও chunk-boundary repair |
| ২. অনুবাদ | **Google Gemini API**; টেক্সট ও নির্দিষ্ট hotword, timing অনুযায়ী ছোট করার অনুরোধ নেই |
| ৩. বাংলা TTS | `edge-tts`, ডিফল্ট `bn-BD-PradeepNeural`; শনাক্ত নীরবতা padding রেখে বাদ |
| ৪. সিঙ্ক ও মাক্স | audio কখনো দ্রুত নয়: ছোট হলে সর্বনিম্ন ০.৯০× ধীর; দীর্ঘ হলে ওই video window-এর চলমান tail repeat; ছোট window-এর ছবি সর্বোচ্চ ১.১৫× দ্রুত |
| ৫. রিপোর্ট | source, edited output ও processing-এর আলাদা প্রতি-সেকেন্ড timeline; retry/fallback ও review flags |

Edit পূর্ণ frame-এ হয়, তাই মিলিসেকেন্ডের বাড়তি সময় পরের frame-এ round হতে পারে; repeated ছবিতে দৃশ্যমান jump হতে পারে। Silence ও ASR heuristic, উচ্চারণ/অর্থ নিশ্চিত নয়। Video re-encode হলে সময় ও resource বাড়তে পারে।

---
### ব্যবহারবিধি
নিচের **৫টি ধাপ** উপর থেকে নিচে একে একে `Shift + Enter` চেপে চালান।


## ধাপ ১: Google Drive মাউন্ট

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ধাপ ২: রিপোজিটরি ক্লোন ও ফোল্ডার প্রস্তুত
Pull ব্যর্থ হলে error থামাবে। আগেই ProXMax import করা runtime-এ নতুন commit এলে **Runtime → Restart session** করে আবার ধাপ ১ থেকে চালান।


In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = "https://github.com/redwan-primeAmax/ProXMax.git"
REPO_DIR = "/content/ProXMax"

if globals().get("_PROXMAX_RESTART_REQUIRED"):
    raise RuntimeError("আগের update load করতে Runtime → Restart session করে ধাপ ১ থেকে চালান।")

loaded = any(name == "proxmax" or name.startswith("proxmax.") for name in sys.modules)
try:
    if not pathlib.Path(REPO_DIR).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        updated = True
    else:
        before = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                                check=True, capture_output=True, text=True).stdout.strip()
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
        after = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                               check=True, capture_output=True, text=True).stdout.strip()
        updated = before != after
except (OSError, subprocess.CalledProcessError) as exc:
    raise RuntimeError("ProXMax clone/pull ব্যর্থ হয়েছে; git error ঠিক করে এই সেল আবার চালান।") from exc

if loaded and updated:
    _PROXMAX_RESTART_REQUIRED = True
    raise RuntimeError("নতুন কোড এসেছে কিন্তু পুরোনো ProXMax import করা আছে। Runtime → Restart session করে ধাপ ১ থেকে চালান।")
if loaded:
    print("ℹ️ ProXMax আগে থেকেই import করা; স্থানীয় কোড বদলে থাকলে Runtime restart করুন।")

os.chdir(REPO_DIR)
pathlib.Path("/content/drive/MyDrive/Video/input").mkdir(parents=True, exist_ok=True)
pathlib.Path("/content/drive/MyDrive/Video/output").mkdir(parents=True, exist_ok=True)
print("✅ বর্তমান ডিরেক্টরি:", os.getcwd())


## ধাপ ৩: ডিপেন্ডেন্সি ইনস্টল ও নির্বাচিত faster-whisper মডেল প্রি-ডাউনলোড
`distil-large-v3` ইংরেজি-only ও ডিফল্ট; `large-v3` multilingual বিকল্প। দুটিই `faster-whisper` checkpoint—কোনোটির গতি বা accuracy সব ভিডিওতে নিশ্চিত নয়। নির্বাচিত মডেল এখন ক্যাশ হয়; ঐচ্ছিক Bengali QA চালালে অন্য multilingual মডেলও পরে প্রয়োজন হতে পারে।


In [ ]:
import os, shutil, subprocess, sys, time

if globals().get("_PROXMAX_RESTART_REQUIRED"):
    raise RuntimeError("Runtime restart করে ধাপ ১ থেকে আবার চালান।")

t0 = time.perf_counter()

# ১. প্রয়োজন হলে FFmpeg install; rubberband filter না থাকলে atempo fallback
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    print("📦 FFmpeg ও ffprobe ইনস্টল হচ্ছে...")
    subprocess.run(["apt-get", "install", "-y", "-qq", "--no-install-recommends", "ffmpeg"], check=True)

# ২. পাইথন প্যাকেজ দ্রুত ইনস্টল (Colab-এ আগে থাকা প্যাকেজ স্কিপ হবে)
print("⚡ পাইথন ডিপেন্ডেন্সি ইনস্টল হচ্ছে...")
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "-r", "requirements.txt",
    ],
    check=True,
)

# ৩. নির্বাচিত Whisper মডেল এখনই ডাউনলোড ও ক্যাশ করা (যাতে ডাবিংয়ের সময় ডাউনলোড না হয়)
if "/content/ProXMax" not in sys.path:
    sys.path.insert(0, "/content/ProXMax")

from proxmax.stt import ensure_whisper_model
from proxmax.hotwords import load_hotwords

WHISPER_MODEL = "distil-large-v3"  # @param ["distil-large-v3", "large-v3"]
print(f"📥 Whisper '{WHISPER_MODEL}' মডেল ক্যাশে প্রস্তুত করা হচ্ছে...")
cached_model_dir = ensure_whisper_model(WHISPER_MODEL)
hotword_count = len(load_hotwords("hotwords.txt"))

print(f"✅ সেটআপ সম্পন্ন ({time.perf_counter() - t0:.1f}s)!")
print(f"   • Whisper মডেল ক্যাশ: {cached_model_dir}")
print(f"   • লোড হওয়া Hotwords (FC Mobile + Players + Events): {hotword_count} টি")

## ধাপ ৪: Gemini API ও মডেল/ইনপুট বাজেট সেট করুন (লাইভ API যাচাই সহ)
১. API key নিন: **https://aistudio.google.com/app/apikey**।
২. নিচের API key বক্সে পেস্ট করুন (অথবা Colab Secrets-এ `GEMINI_API_KEY` রাখুন)।
৩. মডেল বদলান এবং প্রতি request-এর input token cap দিন: default **5000**; চাইলে 2000, 10000, 20000 বা যেকোনো custom সংখ্যা (10000+) লিখতে পারবেন; সর্বনিম্ন 1000।
৪. সেল চালালে API key ও মডেল test হবে। Token cap প্রতি Gemini request-এর জন্য; মডেলের নিজস্ব context limit-এর বেশি পাঠানো যাবে না।


In [ ]:
import os, sys
if globals().get("_PROXMAX_RESTART_REQUIRED"):
    raise RuntimeError("Runtime restart করে ধাপ ১ থেকে আবার চালান।")
if "/content/ProXMax" not in sys.path:
    sys.path.insert(0, "/content/ProXMax")

# @markdown ### 🔑 এখানে আপনার Gemini API Key দিন:
GEMINI_API_KEY = ""  # @param {type:"string"}
# @markdown ### 🤖 অনুবাদের মডেল:
GEMINI_MODEL = "gemini-3.5-flash-lite"  # @param ["gemini-3.5-flash-lite", "gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.0-flash"] {allow-input: true}
# @markdown ### 🧮 প্রতি request-এর সর্বোচ্চ input token (ডিফল্ট 5000; 10000+ custom মানও লেখা যাবে):
GEMINI_MAX_INPUT_TOKENS = 5000  # @param {type:"integer"}
# @markdown ### 🗣️ বাংলা কণ্ঠ বেছে নিন:
# @markdown - `male` = পুরুষ কণ্ঠ (`bn-BD-PradeepNeural`, ডিফল্ট)
# @markdown - `female` = নারী কণ্ঠ (`bn-BD-NabanitaNeural`)
TTS_GENDER = "male"  # @param ["male", "female"]

if not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY") or ""
    except Exception:
        pass

GEMINI_API_KEY = GEMINI_API_KEY.strip()
GEMINI_MAX_INPUT_TOKENS = int(GEMINI_MAX_INPUT_TOKENS)
if GEMINI_MAX_INPUT_TOKENS < 1000:
    raise ValueError("GEMINI_MAX_INPUT_TOKENS অন্তত 1000 হতে হবে।")
if not GEMINI_API_KEY:
    raise ValueError(
        "❌ GEMINI_API_KEY খালি! উপরের বক্সে আপনার Gemini API key পেস্ট করে সেলটি আবার চালান।"
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["PROXMAX_GEMINI_MODEL"] = GEMINI_MODEL.strip() or "gemini-3.5-flash-lite"
os.environ["PROXMAX_GEMINI_MAX_INPUT_TOKENS"] = str(GEMINI_MAX_INPUT_TOKENS)
os.environ["PROXMAX_TTS_GENDER"] = TTS_GENDER.strip() or "male"

# API Key সত্যিই কাজ করছে কি না তা এখনই টেস্ট করা হচ্ছে
from proxmax.translator import verify_gemini_api

print("🔍 Gemini API Key যাচাই করা হচ্ছে...")
check = verify_gemini_api(
    api_key=GEMINI_API_KEY,
    model_name=os.environ["PROXMAX_GEMINI_MODEL"],
)

# ফলো-আপ মডেল পরিবর্তন হলে তা পরিবেশে আপডেট রাখা
os.environ["PROXMAX_GEMINI_MODEL"] = check["active_model"]
voice_label = "পুরুষ (Pradeep)" if TTS_GENDER == "male" else "নারী (Nabanita)"
print(f"✅ Gemini API Key সচল এবং কাজ করছে!")
print(f"   • সক্রিয় মডেল : {check['active_model']}")
print(f"   • টেস্ট অনুবাদ : {check['sample_reply']}")
print(f"   • বাংলা কণ্ঠ   : {voice_label}")

## ধাপ ৫: সব ভিডিও ডাব করুন!
প্রতিটি সফল ভিডিওর পাশে `<video_stem>_report.markdown` তৈরি হয় (আলাদা SRT নয়)। ব্যর্থ হলে report চালু ও output folder writable থাকলে partial failure report-ও লেখা হয়। Source, edited output ও processing-এর clock আলাদা; প্রতি সেকেন্ডের waveform, timestamp ও রেকর্ড করা retry/fallback দেখায়। Final audio আবার decode হওয়ায় অতিরিক্ত সময় ও অস্থায়ী disk লাগে। রিপোর্ট silent error, দৃশ্য, অর্থ বা উচ্চারণ নিশ্চিত করে না; transcript/ভিডিওর নাম থাকতে পারে, শেয়ারের আগে যাচাই করুন।

`FORCE_REPROCESS` চালু করলে একই নামের পুরোনো ভিডিও ও রিপোর্ট পুনর্লিখিত হতে পারে—প্রয়োজনে আগে backup নিন। বন্ধ থাকলে `config.yaml`-এর নিজস্ব `skip_existing` সেটিং অপরিবর্তিত থাকে।


In [ ]:
import os
if globals().get("_PROXMAX_RESTART_REQUIRED"):
    raise RuntimeError("Runtime restart করে ধাপ ১ থেকে আবার চালান।")

# @markdown ### পুরোনো সফল output-ও নতুন করে বানাবেন? (পুনর্লিখিত হবে)
FORCE_REPROCESS = False  # @param {type:"boolean"}

from proxmax.config import AppConfig
from proxmax.pipeline import run_Single_or_batch
from proxmax.utils import setup_logging

config = AppConfig.from_yaml("config.yaml")
if FORCE_REPROCESS:
    config.skip_existing = False  # ডিফল্টে YAML-এর user setting অপরিবর্তিত থাকে

# উপরের সেলগুলোর সেটিংস প্রয়োগ
if globals().get("WHISPER_MODEL"):
    config.whisper_model = WHISPER_MODEL
if os.environ.get("PROXMAX_GEMINI_MODEL"):
    config.gemini_model = os.environ["PROXMAX_GEMINI_MODEL"]
if os.environ.get("PROXMAX_GEMINI_MAX_INPUT_TOKENS"):
    config.gemini_max_input_tokens = int(os.environ["PROXMAX_GEMINI_MAX_INPUT_TOKENS"])
if os.environ.get("PROXMAX_TTS_GENDER"):
    config.tts_gender = os.environ["PROXMAX_TTS_GENDER"]

# চাইলে এখানে যেকোনো সেটিং কাস্টমাইজ করতে পারেন:
# config.input_dir = "/content/drive/MyDrive/Video/input"
# config.output_dir = "/content/drive/MyDrive/Video/output"
# config.skip_existing = False  # পুরোনো আউটপুট থাকলেও নতুন করে ডাব করতে
# config.stt_hotwords = "CustomName1, CustomName2"  # hotwords.txt-এর পাশাপাশি অতিরিক্ত শব্দ
# config.stt_audio_filter = "highpass60"  # পরীক্ষামূলক; accuracy যাচাই করে ব্যবহার করুন
# config.tts_concurrency = 5  # ডিফল্ট 12; Edge সার্ভিসে throttling বেশি হলে কমান
# config.gemini_concurrency = 1  # ডিফল্ট 2; quota/throttling হলে sequential চেষ্টা করুন
# config.min_tempo = 0.90       # speech ধীর হলেও ০.৯০×-এর নিচে নয়
# config.video_max_speed = 1.15  # শুধু picture; সর্বোচ্চ ১.১৫×
# config.video_workers = 2       # ভিডিও re-encode worker; GPU হলে সর্বোচ্চ ২

print(f"ℹ️ skip_existing={config.skip_existing}; পুনর্লিখনের আগে পুরোনো output backup নিন।")
setup_logging(config.log_level)
summary = run_Single_or_batch(config)

print(f"\n🎉 সফল: {len(summary.succeeded)} | স্কিপ: {len(summary.skipped)} | ব্যর্থ: {len(summary.failed)}")